# Semantic Link Labs: The Inspection
### Best Practice Analyzer
> *"Something's bothering me. I don't know what it is yet. But something's not right."* — Adrian Monk
---
Monk never leaves a scene until everything checks out.
The **Best Practice Analyzer (BPA)** is SLL's version of Monk walking your semantic model
and flagging everything that doesn't sit right — naming conventions, unused columns,
missing descriptions, formatting violations, DAX anti-patterns.
By default, the BPA checks more than 60 rules against your semantic model.
The rules come from experts within Microsoft and the Fabric Community, and violations
are organized into five categories: **Performance, DAX Expressions, Error Prevention, Maintenance, and Formatting.**
He will find something. He always does.
| Step | Theme | What We Do |
|---|---|---|
| 1 | The Walk-Through | Run BPA on a single model with default rules |
| 2 | Filing the Report | Persist results to a delta table |
| 3 | Clearing the District | Scan every model in the workspace |
| 4 | The Operations Center | Build a Direct Lake model and report on BPA results |


---
## Setup

In [1]:
%run 00_Setup_Fresh

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 4, Finished, Available, Finished, True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
✅ semantic-link-labs 0.14.0 installed
✅ Workspace    : FabConSQLCon2026
✅ Workspace ID : 21df5422-28b3-46aa-a1c5-ae2cd2a48578
✅ Lakehouse    : EvidenceLocker
✅ Lakehouse ID : a5f11fba-90dd-4722-81a9-d9cc0ebc6fdc
✅ Setup complete — the investigation begins


---
## Step 1 — "The Walk-Through"

> *Monk walks the scene methodically. Every room. Every corner. He doesn't skip anything.*

Run the BPA against a single model with default rules.
Results display inline in the notebook — violations grouped by category and severity.

`extended=True` runs `set_vertipaq_annotations()` first, which feeds real VertiPaq statistics
into the rule engine before evaluation. Rules that check column cardinality, table size,
and memory footprint will use actual numbers rather than estimates.
On small models the output looks identical — the difference shows up on large production models.

In [2]:
# Run BPA with default rules — 60+ rules, inline display
# Tip: extended=True runs set_vertipaq_annotations() first for more accurate
# size/cardinality rules on large production models (return_dataframe=True to get the raw data)

labs.run_model_bpa(dataset=DATASET, workspace=WORKSPACE)

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 5, Finished, Available, Finished, False)

Rule Name,Object Type,Object Name,Severity
Add data category for columnsAdd Data Category property for appropriate columns.,Column,'dim_District'[Latitude],ℹ️
Add data category for columnsAdd Data Category property for appropriate columns.,Column,'dim_District'[Longitude],ℹ️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'dim_Resolution'[ResolutionKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'fact_Incidents'[CallToArrivalMinutes],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'fact_Incidents'[DaysSinceReported],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'fact_Incidents'[IncidentKey],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'fact_Incidents'[OfficerCount],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'fact_Incidents'[ResolutionsID],⚠️
"Do not summarize numeric columnsNumeric columns (integer, decimal, double) should have their SummarizeBy property set to ""None"" to avoid accidental summation in Power BI (create measures instead).",Column,'fact_Incidents'[TimeKey],⚠️
First letter of objects must be capitalizedThe first letter of object names should be capitalized to maintain professional quality.,Partition,'dim_CrimeType'[dim_CrimeType],ℹ️


### The Rule Catalog
`model_bpa_rules()` returns the full set of rules BPA evaluates — name, category, severity, and description.
Worth displaying before you run so the audience can see what they're about to get.

In [3]:

df = labs.model_bpa_rules()
display(df)

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05f669e1-76fe-435e-b220-c2972e0e379d)

### Targeted Scans — Running a Rule Subset
You don't have to run all 60+ rules every time. Filter `model_bpa_rules()` to any subset
and pass it in via the `rules` parameter.
Here we pull two rules that apply directly to the auto-date model (`DATASET_AD`) —
one for the auto-date tables themselves, one for a common DAX filter pattern they introduce.

In [4]:
rules = labs.model_bpa_rules()

__rule_subset = rules[
    (rules['Rule Name'] == "Remove auto-date table") |
    (rules['Rule Name'] == "Filter column values with proper syntax")
].copy()

display(__rule_subset)

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fdb4547d-9fac-4b9b-ab13-261285d00dda)

Run BPA against the auto-date model using only the targeted rule subset.
Cleaner output, faster execution, directly relevant findings.

In [5]:
labs.run_model_bpa(
    dataset=DATASET_AD,
    rules=__rule_subset
    )

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 8, Finished, Available, Finished, False)

Rule Name,Object Type,Object Name,Severity
"Filter column values with proper syntaxInstead of using this pattern FILTER('Table','Table'[Column]=""Value"") for the filter parameters of a CALCULATE or CALCULATETABLE function, use one of the options below. As far as whether to use the KEEPFILTERS function, see the second reference link below. Option 1: KEEPFILTERS('Table'[Column]=""Value"") Option 2: 'Table'[Column]=""Value""",Measure,#LargeSales_Slow,⚠️
Rule Name,Object Type,Object Name,Severity
Remove auto-date tableAvoid using auto-date tables. Make sure to turn off auto-date table in the settings in Power BI Desktop. This will save memory resources.,Table,DateTableTemplate_d5981dd2-e389-41b9-abc2-6f1da89c0152,⚠️
Remove auto-date tableAvoid using auto-date tables. Make sure to turn off auto-date table in the settings in Power BI Desktop. This will save memory resources.,Table,LocalDateTable_1a151f76-9243-495e-8cca-934dd1675648,⚠️
Remove auto-date tableAvoid using auto-date tables. Make sure to turn off auto-date table in the settings in Power BI Desktop. This will save memory resources.,Table,LocalDateTable_6ba37e3b-3ddd-4942-bdf8-35e48aeef5ab,⚠️
Remove auto-date tableAvoid using auto-date tables. Make sure to turn off auto-date table in the settings in Power BI Desktop. This will save memory resources.,Table,LocalDateTable_91db0935-ef50-4984-95fa-39a3fd6e7bae,⚠️
Remove auto-date tableAvoid using auto-date tables. Make sure to turn off auto-date table in the settings in Power BI Desktop. This will save memory resources.,Table,LocalDateTable_b4b67843-943d-4c6b-8ce4-69d8e3deceab,⚠️
Remove auto-date tableAvoid using auto-date tables. Make sure to turn off auto-date table in the settings in Power BI Desktop. This will save memory resources.,Table,LocalDateTable_b56160cf-47a9-4629-a90b-c38e46b47a61,⚠️


---
## Step 2 — "Filing the Report"

> *Monk writes everything down. In the case file. In order. With timestamps.*

Setting `export=True` writes the BPA results to a **`modelbparesults` delta table**
in your attached lakehouse — so violations are persisted, queryable, and trackable over time.

This is where BPA becomes a **governance process** rather than a one-time check.
Run it on a schedule, append results, trend your violations over sprints.

In [6]:
# Export results to delta table in the attached lakehouse
# Appends to 'modelbparesults' — safe to run repeatedly
labs.run_model_bpa(dataset=DATASET, workspace=WORKSPACE, export=True)

print(f"✅ BPA results exported to '{LAKEHOUSE_NAME}/modelbparesults'")

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 9, Finished, Available, Finished, False)

🟢 The dataframe has been saved as the 'modelbparesults' table in the 'EvidenceLocker' lakehouse within the 'FabConSQLCon2026' workspace.
✅ BPA results exported to 'EvidenceLocker/modelbparesults'


### Bonus: Internationalization
Pass a `language` parameter and BPA translates rule names, categories, and descriptions dynamically.
The violations are the same — the output language changes.
Useful for teams working across regions, or just for a good audience reaction.

In [7]:
# Bonus: run in a different language
# Rules, categories and descriptions are dynamically translated
labs.run_model_bpa(dataset=DATASET, workspace=WORKSPACE, language="French")

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 10, Finished, Available, Finished, False)

Rule Name,Object Type,Object Name,Severity
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[Category],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[Copy_Category],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[Copy_Copy_Category],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[CrimeTypeKey],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[SeverityScore],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[SubCategory],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_CrimeType'[UCRCode],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_Date'[DateKey],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_Date'[DayOfWeek],ℹ️
"Objets visibles sans descriptionAjouter des descriptions aux objets. Ces descriptions s’affichent au survol dans la liste des champs dans Power BI Desktop. En outre, vous pouvez utiliser ces descriptions pour créer un dictionnaire de données automatisé.",Column,'dim_Date'[FullDate],ℹ️


---
## Step 3 — "Clearing the District"

> *One case solved. But Monk notices the whole district has the same problem.*

`run_model_bpa_bulk` scans **every semantic model in a workspace** in one call,
appending all results to the same `modelbparesults` delta table.

This is the enterprise play — automated BPA across every model your team owns.
No manual runs. No missed models. Monk inspects the whole building, not just one apartment.

In [8]:
# Scan every semantic model in the current workspace
# Appends all results to 'modelbparesults' delta table

labs.run_model_bpa_bulk(workspace=WORKSPACE_ID)

print("✅ Bulk BPA complete — all models in workspace scanned")

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 11, Finished, Available, Finished, False)

⌛ Collecting Model BPA stats for the 'SFCaseFiles' semantic model within the 'FabConSQLCon2026' workspace.
🟢 Collected Model BPA stats for the 'SFCaseFiles' semantic model within the 'FabConSQLCon2026' workspace.
⌛ Collecting Model BPA stats for the 'Contoso10K_RI' semantic model within the 'FabConSQLCon2026' workspace.
🟢 Collected Model BPA stats for the 'Contoso10K_RI' semantic model within the 'FabConSQLCon2026' workspace.
⌛ Collecting Model BPA stats for the 'Contoso10K_WithAutoDate' semantic model within the 'FabConSQLCon2026' workspace.
🟢 Collected Model BPA stats for the 'Contoso10K_WithAutoDate' semantic model within the 'FabConSQLCon2026' workspace.
⌛ Collecting Model BPA stats for the 'Trudoso10K' semantic model within the 'FabConSQLCon2026' workspace.
🟢 Collected Model BPA stats for the 'Trudoso10K' semantic model within the 'FabConSQLCon2026' workspace.
⌛ Collecting Model BPA stats for the 'Trudoso10K_WithAutoDateXL' semantic model within the 'FabConSQLCon2026' workspace.
🟢

---
## Step 4 — "The Operations Center"
> *Every good detective has a wall. Monk has a very organized wall.*
SLL can build a full **Direct Lake semantic model and Power BI report** on top of your
`modelbparesults` delta table — two function calls.
- `create_model_bpa_semantic_model()` — creates the **BPAModel** semantic model in Direct Lake mode
- `create_model_bpa_report()` — creates the **ModelBPA** Power BI report bound to that model
Since it's Direct Lake, as you append new BPA runs the report updates automatically —
no model refresh required. Run `run_model_bpa_bulk()` on a schedule and the wall updates itself.
*Monk approves.*

In [9]:
# Step 1 — Create the BPA semantic model (Direct Lake on modelbparesults)
labs.create_model_bpa_semantic_model()

print("BPAModel semantic model created in Direct Lake mode")

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 12, Finished, Available, Finished, False)

🟢 The 'ModelBPA' semantic model was created within the 'FabConSQLCon2026' workspace.
🟢 The 'BPAResults' table has been added to the 'ModelBPA' semantic model within the 'FabConSQLCon2026' workspace.
🟢 The 'modelbparesults' partition has been added to the 'BPAResults' table in the 'ModelBPA' semantic model within the 'FabConSQLCon2026' workspace.
🟢 The 'Capacity_Name' column has been added to the 'BPAResults' table as a 'String' data type in the 'ModelBPA' semantic model within the 'FabConSQLCon2026' workspace.
🟢 The 'Capacity_Id' column has been added to the 'BPAResults' table as a 'String' data type in the 'ModelBPA' semantic model within the 'FabConSQLCon2026' workspace.
🟢 The 'Workspace_Name' column has been added to the 'BPAResults' table as a 'String' data type in the 'ModelBPA' semantic model within the 'FabConSQLCon2026' workspace.
🟢 The 'Workspace_Id' column has been added to the 'BPAResults' table as a 'String' data type in the 'ModelBPA' semantic model within the 'FabConSQLCo

In [10]:
# Step 2 — Create the BPA Power BI report
rep.create_model_bpa_report()

print("BPAReport created")
print(f"Navigate to your workspace to open the report and view results")
print("Going forward just run 'run_model_bpa_bulk' — results appear automatically")

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 13, Finished, Available, Finished, True)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/sempy/fabric/_credentials.py:239: FutureWarning: The 'type' parameter is deprecated and will be removed in a future version. Use 'item_type' instead.
  return func(*args, **kwargs)


🟢 The 'ModelBPA' report within the 'FabConSQLCon2026' workspace has been successfully updated.
BPAReport created
Navigate to your workspace to open the report and view results
Going forward just run 'run_model_bpa_bulk' — results appear automatically


### Launch the Report
`launch_report()` opens the ModelBPA report directly from the notebook.
The report is already populated — Step 3 appended results to `modelbparesults`
and the Direct Lake model picked them up automatically.

In [11]:
rep.launch_report(report="ModelBPA")

StatementMeta(, 5e561522-de02-499e-9f66-adf495eaab61, 14, Finished, Available, Finished, True)

Report()

---
## Ongoing Workflow

Once the model and report are built, the ongoing workflow is just one cell:

```python
# Run this on a schedule — weekly, per sprint, pre-release
labs.run_model_bpa_bulk(workspace=WORKSPACE)
```

Results append to `modelbparesults`, the Direct Lake model picks them up automatically,
and the report is always current. No refresh. No manual steps.

**Put it in a Fabric pipeline and you have automated model governance.**

*Monk would run it every day. Probably twice.*

---
## Wrap-Up

| Step | Function | What it does |
|---|---|---|
| 1 | `run_model_bpa()` | Single model — 60+ rules, inline display |
| 1 | `model_bpa_rules()` | Inspect and filter the rule catalog |
| 2 | `run_model_bpa(export=True)` | Persist results to `modelbparesults` delta table |
| 3 | `run_model_bpa_bulk()` | Scan every model in the workspace |
| 4 | `create_model_bpa_semantic_model()` | Build Direct Lake model on BPA results |
| 4 | `create_model_bpa_report()` | Build Power BI report on BPA model |
| 4 | `launch_report()` | Open the report directly from the notebook |

### Resources:
- [SLL GitHub — BPA Report notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Best%20Practice%20Analyzer%20Report.ipynb)
- [SLL GitHub — Model Optimization notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Model%20Optimization.ipynb)

> *"You know what I hate? Unformatted measures.*
> *You know what else I hate? Everything else.*" — Adrian Monk (probably)
